# Cluster 1 Usage

This notebook demonstrates how to load and use the exported predictor for cluster 1.

The predictor is a dict-based bundle containing:
- selected_features: 6 financial features picked via L1 logistic regression
- stacking_model: fitted StackingClassifier (LR + RF + GB base models, LR meta-learner)
- threshold: tuned decision threshold (with sparsity guard for the team budget)
- n_features: feature count for Table 3 reporting

## Setup

In [2]:
import pandas as pd
import joblib

# Import helper functions
from aania_cluster_classes import predict_with_bundle, predict_proba_with_bundle

# Load the bundle
c1_bundle = joblib.load('aania_cluster1_model.joblib')

print('Cluster 1 predictor loaded')
print(f"  Features:  {c1_bundle['n_features']}")
print(f"  Threshold: {c1_bundle['threshold']:.2f}")

Cluster 1 predictor loaded
  Features:  6
  Threshold: 0.40


## Selected features


In [3]:
for f in c1_bundle['selected_features']:
    print(f'  - {f.strip()}')

  - Tax rate (A)
  - Borrowing dependency
  - Average Collection Days
  - Allocation rate per person
  - Cash Turnover Rate
  - Net Income to Stockholder's Equity


## Load Data

In [4]:
df = pd.read_csv('../../Clusters/cluster_1.csv')
X = df.drop(columns=['Bankrupt?'])
y = df['Bankrupt?']

## Make Predictions

In [5]:
y_pred = predict_with_bundle(c1_bundle, X)

print(f'Predictions made for {len(y_pred)} samples')
print(f'\nPrediction distribution:')
print(f'  Non-Bankrupt (0): {(y_pred == 0).sum()}')
print(f'  Bankrupt (1): {(y_pred == 1).sum()}')

Predictions made for 1749 samples

Prediction distribution:
  Non-Bankrupt (0): 1540
  Bankrupt (1): 209


## Sanity check against training notebook

Should match the Table 3 numbers from Aania_C1.ipynb (TT=8, TF=0).

In [7]:
TT = ((y == 1) & (y_pred == 1)).sum()
TF = ((y == 1) & (y_pred == 0)).sum()
print(f'TT={TT}, TF={TF}, N_features={c1_bundle["n_features"]}')

TT=8, TF=0, N_features=6
